In [ ]:
import sys
sys.path.append("/path/to/example/autocompute/static")
from g16_env import load_gaussian_env
load_gaussian_env()

In [ ]:
import subprocess
import sys
import os
import re
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from openbabel import openbabel
from openbabel import pybel
import shutil
import time
import pandas as pd
import glob
from rdkit import Chem
import signal
from openpyxl import Workbook, load_workbook
from openpyxl.utils import get_column_letter
from sklearn.linear_model import LinearRegression
from datetime import datetime, timedelta
from rdkit.Chem import rdMolDescriptors

In [ ]:

try:
    from openpyxl import load_workbook
except ImportError:
    print("Public message removed for release.")
    exit()

In [ ]:

def get_filename_without_extension(xlsx_path):
    
    
    base_name = os.path.basename(xlsx_path)
    
    file_name_without_extension = os.path.splitext(base_name)[0]
    return file_name_without_extension

In [ ]:

def extract_energy_from_out(file_path):
    energy = None
    with open(file_path, 'r') as file:
        lines = file.readlines()
        energy_line = ''
        for i, line in enumerate(lines):
            
            line = line.strip().replace('\n', '')
            
            if line.strip().endswith('H') and i+1 < len(lines):
                next_line = lines[i+1]
                if next_line.strip().startswith('F='):
                    
                    line = line.strip() + next_line.strip()
            
            if line.strip().endswith('HF') and i+1 < len(lines):
                next_line = lines[i+1]
                if next_line.strip().startswith('='):
                    
                    line = line.strip() + next_line.strip()
            
            if 'HF=' in line:
                
                start = line.find('HF=') + 3  
                energy_line = line[start:]
                
                if '\\' in energy_line:
                    energy = energy_line.split('\\')[0].strip()
                    return energy
                else:
                    
                    for j in range(i+1, len(lines)):
                        energy_line += lines[j].strip()
                        if '\\' in lines[j]:
                            energy = energy_line.split('\\')[0].strip()
                            return energy
    return energy  

In [ ]:
def extract_dipole_moment(file_path):
    
    try:
        with open(file_path, 'r') as file:
            lines = file.readlines()  

        
        dipole_moment_lines = [
            i for i, line in enumerate(lines)
            if re.search(r"Dipole moment \(field-independent basis, Debye\):", line)
        ]

        
        if not dipole_moment_lines:
            print("Public message removed for release.")
            return None

        
        last_line_number = dipole_moment_lines[-1]

        
        next_line = lines[last_line_number + 1]
        match = re.search(r"Tot=\s*(-?\d+\.\d+)", next_line)

        
        if match:
            return float(match.group(1))
        else:
            print("Public message removed for release.")
            return None

    except FileNotFoundError:
        print("Public message removed for release.")
        return None
    except Exception as e:
        print("Public message removed for release.")
        return None

In [ ]:

def extract_homo_lumo(file_path):
    
    try:
        filename = get_filename_without_extension(file_path)
        
        with open(file_path, 'r') as file:
            content = file.readlines()
        
        
        pop_analysis_lines = [i for i, line in enumerate(content) if re.match(r'\s*Population analysis', line)]
        
        
        if len(pop_analysis_lines) == 0:
            raise ValueError("Public message removed for release.")

        
        last_population_analysis = pop_analysis_lines[-1]

        
        homo, lumo = None, None
        for line_number in range(last_population_analysis, len(content)):
            occ_match = re.match(r'\s*Alpha\s+occ.\s+eigenvalues\s+--(.*)', content[line_number])
            virt_match = re.match(r'\s*Alpha\s+virt.\s+eigenvalues\s+--(.*)', content[line_number + 1]) if line_number + 1 < len(content) else None
            
            if occ_match and virt_match:
                
                homo = float(occ_match.group(1).strip().split()[-1])  
                lumo = float(virt_match.group(1).strip().split()[0])  
                break
        
        
        if homo is None or lumo is None:
            raise ValueError("Public message removed for release.")
        
        
        return homo, lumo

    except FileNotFoundError:
        
        raise FileNotFoundError("Public message removed for release.")
    except Exception as e:
        
        raise Exception("Public message removed for release.")

In [ ]:

def extract_entropy(file_path):
    
    entropy_pattern = re.compile(r'\s*Total\s+[0-9.-]+\s+[0-9.-]+\s+([0-9.-]+)')
    try:
        with open(file_path, 'r') as file:
            file_content = file.read()  
            match = entropy_pattern.search(file_content)
            if match:
                
                entropy_value_cal = float(match.group(1))
                
                entropy_value_joules = entropy_value_cal * 4.184 
                return entropy_value_joules
    except FileNotFoundError:
        print(f"The file {file_path} was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

    return None  

In [ ]:

def extract_enthalpy_correction(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            if "Thermal correction to Enthalpy" in line:
                
                match = re.search(r"=\s*([-\d.]+)", line)
                if match:
                    return float(match.group(1))
    return None

In [ ]:

def extract_gibbs_correction(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            if "Thermal correction to Gibbs Free Energy" in line:
                
                match = re.search(r"=\s*([-\d.]+)", line)
                if match:
                    return float(match.group(1))
    return None

In [ ]:

def extract_gibbs_correction(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            if "Thermal correction to Gibbs Free Energy" in line:
                
                match = re.search(r"=\s*([-\d.]+)", line)
                if match:
                    return float(match.group(1))
    return None

In [ ]:

def compute_properties(df, opt_success_dir, energy_success_dir):
    
    
    reactant_name_cols = [col for col in df.columns if col.startswith('Reactant_Name_')]
    product_name_cols = [col for col in df.columns if col.startswith('Product_Name_')]

    
    properties_cols = ['Properties_' + col for col in reactant_name_cols + product_name_cols]
    for col in properties_cols:
        df[col] = 0.0  

    
    for index, row in df.iterrows():
        
        reactant_names = [row[col] for col in reactant_name_cols if pd.notna(row[col])]
        product_names = [row[col] for col in product_name_cols if pd.notna(row[col])]
        all_molecule_names = reactant_names + product_names

        
        reaction_index_file = str(index)

        
        energy_dest_dir = os.path.join(energy_success_dir, reaction_index_file)

        
        for molecule_name in all_molecule_names:
            
            opt_out_file = os.path.join(opt_success_dir, molecule_name + '.out')

            
            energy_out_file = os.path.join(energy_dest_dir, molecule_name + '.out')

            
            if not os.path.exists(opt_out_file):
                print("Public message removed for release.")
                continue
            
            if not os.path.exists(energy_out_file):
                print("Public message removed for release.")
                continue

            
            G_correction = float(extract_gibbs_correction(opt_out_file) or 0.0)  
            H_correction = float(extract_enthalpy_correction(opt_out_file) or 0.0)  
            S = float(extract_entropy(opt_out_file) or 0.0)  
            HOMO, LUMO = extract_homo_lumo(opt_out_file)  
            HOMO = float(HOMO or 0.0)  
            LUMO = float(LUMO or 0.0)  
            Dipole = float(extract_dipole_moment(opt_out_file) or 0.0)  
            Energy = float(extract_energy_from_out(energy_out_file) or 0.0)  
            
            
            G = G_correction + Energy
            H = H_correction + Energy

            
            properties_dict = {
                'G_correction': G_correction,
                'H_correction': H_correction,
                'S': S,
                'HOMO': HOMO,
                'LUMO': LUMO,
                'Dipole': Dipole,
                'Energy': Energy,
                'G': G,
                'H': H
            }

            
            
            if molecule_name in reactant_names:
                col_index = reactant_names.index(molecule_name)
                col_name = 'Properties_' + reactant_name_cols[col_index]
            elif molecule_name in product_names:
                col_index = product_names.index(molecule_name)
                col_name = 'Properties_' + product_name_cols[col_index]
            else:
                print("Public message removed for release.")
                continue

            
            df.at[index, col_name] = str(properties_dict)

    
    return df

In [ ]:

def calculate_reaction_thermo(df):
    
    import pandas as pd
    import ast  

    
    reactant_name_cols = [col for col in df.columns if col.startswith('Reactant_Name_')]
    product_name_cols = [col for col in df.columns if col.startswith('Product_Name_')]

    
    reactant_coefficient_cols = [col for col in df.columns if col.startswith('Reactant_Coefficient_')]
    product_coefficient_cols = [col for col in df.columns if col.startswith('Product_Coefficient_')]

    
    properties_cols = [col for col in df.columns if col.startswith('Properties_')]

    
    df['Reaction_G (kJ/mol)'] = None
    df['Reaction_H (kJ/mol)'] = None
    

    
    for index, row in df.iterrows():
        
        total_G_reactants = 0.0
        total_H_reactants = 0.0
        total_S_reactants = 0.0

        total_G_products = 0.0
        total_H_products = 0.0
        total_S_products = 0.0

        
        for name_col, coeff_col in zip(reactant_name_cols, reactant_coefficient_cols):
            molecule_name = row[name_col]
            coefficient = row[coeff_col]

            
            if pd.isna(molecule_name) or pd.isna(coefficient):
                continue  

            coefficient = float(coefficient)  

            
            properties_col = 'Properties_' + name_col

            
            properties_str = row[properties_col]
            if pd.isna(properties_str):
                print("Public message removed for release.")
                continue

            
            try:
                properties_dict = ast.literal_eval(properties_str)
            except:
                print("Public message removed for release.")
                continue

            
            G = properties_dict.get('G', None)
            H = properties_dict.get('H', None)
            S = properties_dict.get('S', None)

            if G is None or H is None or S is None:
                print("Public message removed for release.")
                continue

            
            G = float(G)
            H = float(H)
            S = float(S)

            
            total_G_reactants += G * coefficient
            total_H_reactants += H * coefficient
            total_S_reactants += S * coefficient

        
        for name_col, coeff_col in zip(product_name_cols, product_coefficient_cols):
            molecule_name = row[name_col]
            coefficient = row[coeff_col]

            
            if pd.isna(molecule_name) or pd.isna(coefficient):
                continue  

            coefficient = float(coefficient)  

            
            properties_col = 'Properties_' + name_col

            
            properties_str = row[properties_col]
            if pd.isna(properties_str):
                print("Public message removed for release.")
                continue

            
            try:
                properties_dict = ast.literal_eval(properties_str)
            except:
                print("Public message removed for release.")
                continue

            
            G = properties_dict.get('G', None)
            H = properties_dict.get('H', None)
            S = properties_dict.get('S', None)

            if G is None or H is None or S is None:
                print("Public message removed for release.")
                continue

            
            G = float(G)
            H = float(H)
            S = float(S)

            
            total_G_products += G * coefficient
            total_H_products += H * coefficient
            total_S_products += S * coefficient

        
        delta_G = total_G_products - total_G_reactants
        delta_H = total_H_products - total_H_reactants
        delta_S = total_S_products - total_S_reactants

        
        df.at[index, 'Reaction_G (kJ/mol)'] = delta_G * 2625.5
        df.at[index, 'Reaction_H (kJ/mol)'] = delta_H * 2625.5
        

    
    return df

In [ ]:
mol_excel_file = "HTQC_reaction_thermo.xlsx"
opt_success_dir = "opt+freq/success"
energy_success_dir = "energy/success"

df = pd.read_excel(mol_excel_file)

df = compute_properties(df, opt_success_dir, energy_success_dir)

df = calculate_reaction_thermo(df)
df.to_excel(mol_excel_file,index=None)